# Araba Fiyatları (Car Prices)

🎯 Bu challenge’ın amacı, bir dataset hazırlamak ve şimdiye kadar öğrendiğiniz bazı feature selection tekniklerini uygulamaktır.

🚗 Arabalarla ilgili bir veri setiyle çalışıyoruz ve bir arabanın pahalı mı yoksa ucuz mu olduğunu tahmin etmek istiyoruz.

In [108]:
# Data manipulation
import numpy as np
import pandas as pd
# Data visualisation
import matplotlib.pyplot as plt
import seaborn as sns
# Sayısal bir özelliğin normal dağılım gösterip göstermediğini kontrol etme
from statsmodels.graphics.gofplots import qqplot


In [135]:
url = "https://d32aokrjazspmn.cloudfront.net/materials/ML_Cars_dataset.csv"

❓ CSV dosyasını `df` adlı bir veri çerçevesine yükleyin.

In [147]:
df = pd.read_csv(url)
df = df.copy()

ℹ️ Dataset’in açıklaması [burada](https://drive.google.com/file/d/1ADSyjWfRGYqdXwCCN4PPC7PjQeMZ-ap-/view?usp=sharing ) mevcuttur. Egzersiz boyunca buna mutlaka referans verin.

## (1) Yinelenenler (Duplicates)

❓ Varsa, veri kümesinden yinelenenleri kaldırın. ❓

*Veri çerçevesini `df`* üzerine yazın.

In [148]:
df = df.drop_duplicates()
df

,aspiration,enginelocation,carwidth,curbweight,enginetype,cylindernumber,stroke,peakrpm,price
0,std,front,64.1,2548,dohc,four,2.68,5000,expensive
2,std,front,65.5,2823,ohcv,six,3.47,5000,expensive
3,std,front,NaN,2337,ohc,four,3.40,5500,expensive
4,std,front,66.4,2824,ohc,five,3.40,5500,expensive
5,std,front,66.3,2507,ohc,five,3.40,5500,expensive
...,...,...,...,...,...,...,...,...,...
200,std,front,68.9,2952,ohc,four,3.15,5400,expensive
201,turbo,front,68.8,3049,ohc,four,3.15,5300,expensive
202,std,front,68.9,3012,ohcv,six,2.87,5500,expensive
203,turbo,front,68.9,3217,ohc,six,3.40,4800,expensive


## (2)  Eksik değerler (Missing values)

❓ Eksik değerleri bulun ve bunları ya `strategy = "most frequent"` (kategorik değişkenler için) ya da `strategy = "median"` (sayısal değişkenler için) kullanarak doldurun ❓

In [149]:
df.isnull().sum()
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

In [150]:
from sklearn.impute import SimpleImputer

# 1️⃣ Sayısal imputer
num_imputer = SimpleImputer(strategy='median')
num_array = num_imputer.fit_transform(df[num_cols])
df[num_cols] = pd.DataFrame(num_array, columns=num_cols, index=df.index).astype(float)

# 2️⃣ Kategorik imputer
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_array = cat_imputer.fit_transform(df[cat_cols])
df[cat_cols] = pd.DataFrame(cat_array, columns=cat_cols, index=df.index)

# Kontrol
print(df[num_cols].dtypes)
print(df[num_cols].isna().sum())

curbweight    float64
stroke        float64
peakrpm       float64
dtype: object
curbweight    0
stroke        0
peakrpm       0
dtype: int64


### `carwidth`

<details>
    <summary> 💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>carwidth</code> sütununda eksik değerler birden fazla şekilde temsil edilmektedir. Bazıları <code>np.nan</code>, bazıları ise <code>*</code> olarak yer alır. Bunlar tespit edildikten sonra, eksik değerler verinin %30’undan daha azını oluşturduğu için medyan değerle doldurulabilir.
</details>

In [153]:
import numpy as np
import pandas as pd

# 1. '*' değerlerini NaN ile değiştir
df["carwidth"] = df["carwidth"].replace("*", np.nan)

# 2. Sayısal tipe çevir (Hatalı karakterleri NaN yapar)
# pd.to_numeric kullanımı çıktı tipini otomatik float/int yapmaya zorlar
df["carwidth"] = pd.to_numeric(df["carwidth"], errors='coerce')

# 3. Boşlukları (NaN) median ile doldur
median_value = df["carwidth"].median()
df["carwidth"] = df["carwidth"].fillna(median_value)

# 4. Tip kontrolünü garantiye al (Sütunu tamamen float yap)
df["carwidth"] = df["carwidth"].astype(float)

# Sonuç Kontrolü
print(f"Veri Tipi: {df['carwidth'].dtype}")
print(f"Kalan Eksik Değer: {df['carwidth'].isna().sum()}")

Veri Tipi: float64
Kalan Eksik Değer: 0


### `enginelocation`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>enginelocation</code> kategorik bir feature olduğundan ve kategorilerin büyük çoğunluğu <code>front</code> olduğu için, en sık görülen değerle doldurun.
</details>

In [156]:
#yukarıda yapmış olduğum sekilde de yapılabilir ayrıca bu sekilde tekil olarakda:
most_frequent = df["enginelocation"].mode()[0]
df["enginelocation"] = df["enginelocation"].fillna(most_frequent)

🧪 **Kodunu test et**

In [157]:
from nbresult import ChallengeResult

result = ChallengeResult('missing_values',
                         dataset = df)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/aybukealtuntas/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/aybukealtuntas/S16D2-S-Data-car-prices/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 2 items

test_missing_values.py::TestMissing_values::test_carwidth PASSED         [ 50%]
test_missing_values.py::TestMissing_values::test_engine_location PASSED  [100%]

============================== 2 passed in 0.55s ===============================


💯 You can commit your code:

git add tests/missing_values.pickle

git commit -m 'Completed missing_values step'

git push origin master



## (3) Sayısal özelliklerin ölçeklendirilmesi (Scaling the numerical features)

In [158]:
# Hatırlatma olarak, DataFrame hakkında bazı bilgiler
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 191 entries, 0 to 204
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   aspiration      191 non-null    object 
 1   enginelocation  191 non-null    object 
 2   carwidth        191 non-null    float64
 3   curbweight      191 non-null    float64
 4   enginetype      191 non-null    object 
 5   cylindernumber  191 non-null    object 
 6   stroke          191 non-null    float64
 7   peakrpm         191 non-null    float64
 8   price           191 non-null    object 
dtypes: float64(4), object(5)
memory usage: 14.9+ KB


In [159]:
# Ve işte ölçeklendirmemiz gereken veri kümesinin sayısal özellikleri
numerical_features = df.select_dtypes(exclude=['object']).columns
numerical_features

Index(['carwidth', 'curbweight', 'stroke', 'peakrpm'], dtype='object')

❓ **Soru: Sayısal feature’ların ölçeklenmesi** ❓

Sayısal feature’ları aykırı değerler (outliers) ve dağılımları açısından inceleyin ve duruma göre aşağıdaki yöntemleri uygulayın:
- Robust Scaler
- Standard Scaler

Dönüştürülmüş değerlerle orijinal sütunları değiştirin.

### `peakrpm` , `carwidth` , & `stroke`

<details>
    <summary>💡 <i>İpucu</i> </summary>

    
ℹ️ <code>peakrpm</code>, <code>carwidth</code> ve <code>stroke</code> normal dağılıma sahiptir ancak aynı zamanda bazı aykırı değerler (outlier) içerir. Bu nedenle `RobustScaler()` kullanılması tavsiye edilir.
</details>

In [160]:
from sklearn.preprocessing import RobustScaler
import pandas as pd

cols = ["peakrpm", "carwidth", "stroke"]

# 1️⃣ float64 olduğundan emin ol (loc ile)
df[cols] = df[cols].astype(float)

# 2️⃣ Scaler
scaler = RobustScaler()
scaled_array = scaler.fit_transform(df[cols])

# 3️⃣ DataFrame yap ve kolon isimlerini koru
scaled_df = pd.DataFrame(scaled_array, columns=cols, index=df.index)

df[cols] = scaled_df

# Kontrol
print(df[cols].dtypes)   # hepsi float64 olmalı
print(df[cols].head())

peakrpm     float64
carwidth    float64
stroke      float64
dtype: object
    peakrpm  carwidth    stroke
0 -0.142857 -0.518519 -2.033333
2 -0.142857  0.000000  0.600000
3  0.571429  0.370370  0.366667
4  0.571429  0.333333  0.366667
5  0.571429  0.296296  0.366667


In [161]:
df[numerical_features].head()
df

,aspiration,enginelocation,carwidth,curbweight,enginetype,cylindernumber,stroke,peakrpm,price
0,std,front,-0.518519,2548.0,dohc,four,-2.033333,-0.142857,expensive
2,std,front,0.000000,2823.0,ohcv,six,0.600000,-0.142857,expensive
3,std,front,0.370370,2337.0,ohc,four,0.366667,0.571429,expensive
4,std,front,0.333333,2824.0,ohc,five,0.366667,0.571429,expensive
5,std,front,0.296296,2507.0,ohc,five,0.366667,0.571429,expensive
...,...,...,...,...,...,...,...,...,...
200,std,front,1.259259,2952.0,ohc,four,-0.466667,0.428571,expensive
201,turbo,front,1.222222,3049.0,ohc,four,-0.466667,0.285714,expensive
202,std,front,1.259259,3012.0,ohcv,six,-1.400000,0.571429,expensive
203,turbo,front,1.259259,3217.0,ohc,six,0.366667,-0.428571,expensive


### `curbweight`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>curbweight</code> normal bir dağılıma sahiptir ve aykırı değer (outlier) içermez. Bu nedenle Standard Scaler ile ölçeklenebilir.
</details>

In [162]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Önce float64 olduğundan emin ol
df[["curbweight"]] = df[["curbweight"]].astype(float)

# Scale et ve tekrar ata
df[["curbweight"]] = scaler.fit_transform(df[["curbweight"]])

In [163]:
df[numerical_features].head()
df

,aspiration,enginelocation,carwidth,curbweight,enginetype,cylindernumber,stroke,peakrpm,price
0,std,front,-0.518519,-0.048068,dohc,four,-2.033333,-0.142857,expensive
2,std,front,0.000000,0.476395,ohcv,six,0.600000,-0.142857,expensive
3,std,front,0.370370,-0.450474,ohc,four,0.366667,0.571429,expensive
4,std,front,0.333333,0.478302,ohc,five,0.366667,0.571429,expensive
5,std,front,0.296296,-0.126260,ohc,five,0.366667,0.571429,expensive
...,...,...,...,...,...,...,...,...,...
200,std,front,1.259259,0.722416,ohc,four,-0.466667,0.428571,expensive
201,turbo,front,1.222222,0.907408,ohc,four,-0.466667,0.285714,expensive
202,std,front,1.259259,0.836844,ohcv,six,-1.400000,0.571429,expensive
203,turbo,front,1.259259,1.227807,ohc,six,0.366667,-0.428571,expensive


🧪 **Kodunu test et**

In [164]:
from nbresult import ChallengeResult

result = ChallengeResult('scaling',
                         dataset = df
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/aybukealtuntas/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/aybukealtuntas/S16D2-S-Data-car-prices/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 4 items

test_scaling.py::TestScaling::test_carwidth PASSED                       [ 25%]
test_scaling.py::TestScaling::test_curbweight PASSED                     [ 50%]
test_scaling.py::TestScaling::test_peakrpm PASSED                        [ 75%]
test_scaling.py::TestScaling::test_stroke PASSED                         [100%]

============================== 4 passed in 0.52s ===============================


💯 You can commit your code:

git add tests/scaling.pickle

git commit -m 'Completed scaling step'

git push origin master



## (4) Kategorik özelliklerin kodlanması (Encoding the categorical features)

❓ **Soru: Kategorik değişkenlerin encode edilmesi** ❓

👇 Encode edilmesi gereken feature’ları inceleyin ve duruma göre aşağıdaki teknikleri uygulayın:

- One-hot encoding
- Manuel ordinal encoding

DataFrame içinde, orijinal feature’ları encode edilmiş versiyonlarıyla değiştirin.

### `aspiration` & `enginelocation`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>aspiration</code> ve <code>enginelocation</code> ikili (binary) kategorik feature’lardır.
</details>

In [165]:
print(df["enginelocation"].unique())
print(df["aspiration"].unique())
# front = 0, rear = 1
df["enginelocation"] = df["enginelocation"].map({"front": 0, "rear": 1})
# std = 0, turbo = 1
df["aspiration"] = df["aspiration"].map({"std": 0, "turbo": 1})
print(df["enginelocation"].unique())
print(df["aspiration"].unique())

['front' 'rear']
['std' 'turbo']
[0 1]
[0 1]


### `enginetype`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>enginetype</code> çok kategorili (multicategorical) bir feature’dır ve One-hot encoding uygulanmalıdır.
</details>

In [166]:
import pandas as pd

# One-hot encoding
df = pd.get_dummies(df, columns=["enginetype"], prefix="enginetype")

# Sadece one-hot kolonlarını int tipine çevir
onehot_cols = df.filter(like="enginetype").columns
df[onehot_cols] = df[onehot_cols].astype(int)

# Kontrol edelim
print(df[onehot_cols].head())

   enginetype_dohc  enginetype_dohcv  enginetype_l  enginetype_ohc  \
0                1                 0             0               0   
2                0                 0             0               0   
3                0                 0             0               1   
4                0                 0             0               1   
5                0                 0             0               1   

   enginetype_ohcf  enginetype_ohcv  enginetype_rotor  
0                0                0                 0  
2                0                1                 0  
3                0                0                 0  
4                0                0                 0  
5                0                0                 0  


In [167]:
df.shape

(191, 15)

### `cylindernumber`

<details>
    <summary>💡 İpucu </summary>

ℹ️ <code>cylindernumber</code> sıralı (ordinal) bir feature’dır ve sayısal değerlere manuel olarak encode edilmelidir.

</details>

In [168]:
print(df["cylindernumber"].unique())
df["cylindernumber"] = df["cylindernumber"].map({"four": 4, "six": 6, "five":5, "three":3, "twelve":12, "two":2, "eight":8})

['four' 'six' 'five' 'three' 'twelve' 'two' 'eight']


❓ Artık `cylindernumber`’ı 2 ile 12 arasında sayısal bir feature’a dönüştürdüğünüze göre, bunu ölçeklendirmeniz gerekiyor ❓

<br/>

<details>
    <summary>💡 İpucu </summary>

`cylindernumber`’ın mevcut dağılımına bakın ve kendinize şu soruları sorun:
- Ölçekleme, bir feature’ın dağılımını etkiler mi?
- Bu feature’ın dağılımına göre en uygun ölçekleme yöntemi hangisidir?
</details>

In [169]:
print(df["cylindernumber"].value_counts())

cylindernumber
4     147
6      23
5      11
8       5
2       3
3       1
12      1
Name: count, dtype: int64


<details>
<summary><i>Ölçekleme ve encoding işlemlerinden sonra DataFrame’inizin nasıl görünmesi gerektiğine dair bir ekran görüntüsü aşağıdadır</i></summary>
    
    
<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/02-Prepare-the-dataset/car_price_after_scaling_and_encoding.png">    

</details>

### `price`

👇 Hedef `price`ı kodlayın.

<details>
    <summary>💡 İpucu </summary>
    <br>
    ℹ️ <code>price</code> target değişkendir ve LabelEncoder ile encode edilmelidir.
</details>

In [170]:
df["price"].unique()
from sklearn.preprocessing import LabelEncoder

# LabelEncoder nesnesi oluştur
le = LabelEncoder()

# price sütununu encode et
df["price"] = le.fit_transform(df["price"])

# Kontrol
print(df["price"].unique())   # [0 1] gibi olmalı
df

[1 0]


,aspiration,enginelocation,carwidth,curbweight,cylindernumber,stroke,peakrpm,price,enginetype_dohc,enginetype_dohcv,enginetype_l,enginetype_ohc,enginetype_ohcf,enginetype_ohcv,enginetype_rotor
0,0,0,-0.518519,-0.048068,4,-2.033333,-0.142857,1,1,0,0,0,0,0,0
2,0,0,0.000000,0.476395,6,0.600000,-0.142857,1,0,0,0,0,0,1,0
3,0,0,0.370370,-0.450474,4,0.366667,0.571429,1,0,0,0,1,0,0,0
4,0,0,0.333333,0.478302,5,0.366667,0.571429,1,0,0,0,1,0,0,0
5,0,0,0.296296,-0.126260,5,0.366667,0.571429,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,0,0,1.259259,0.722416,4,-0.466667,0.428571,1,0,0,0,1,0,0,0
201,1,0,1.222222,0.907408,4,-0.466667,0.285714,1,0,0,0,1,0,0,0
202,0,0,1.259259,0.836844,6,-1.400000,0.571429,1,0,0,0,0,0,1,0
203,1,0,1.259259,1.227807,6,0.366667,-0.428571,1,0,0,0,1,0,0,0


🧪 **Kodunu test et**

In [171]:
from nbresult import ChallengeResult

result = ChallengeResult('encoding',
                         dataset = df)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/aybukealtuntas/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/aybukealtuntas/S16D2-S-Data-car-prices/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 4 items

test_encoding.py::TestEncoding::test_aspiration PASSED                   [ 25%]
test_encoding.py::TestEncoding::test_enginelocation PASSED               [ 50%]
test_encoding.py::TestEncoding::test_enginetype PASSED                   [ 75%]
test_encoding.py::TestEncoding::test_price PASSED                        [100%]

============================== 4 passed in 0.43s ===============================


💯 You can commit your code:

git add tests/encoding.pickle

git commit -m 'Completed encoding step'

git push origin master



## (5) Temel Modelleme (Base Modelling)

👏 Veri kümesi ön işleme tabi tutuldu ve artık modele uyarlanmaya hazır. 

❓ **Soru: Bir classification modelini ilk kez değerlendirme** ❓

Ön işlenmiş bu dataset üzerinde bir `LogisticRegression` modeli için cross-validation çalıştırın ve elde edilen skoru `base_model_score` adlı değişkende saklayın.

In [186]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
X = df.drop(columns=["price"])
y = df["price"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
base_model_score = cross_val_score(model,
                                   X_train,
                                   y_train,
                                   cv=5,
                                   scoring='accuracy') 

print(base_model_score)
print("Average CV accuracy:", base_model_score.mean())
base_model_score = base_model_score.mean()
model.fit(X_train,y_train)

[0.70967742 0.80645161 1.         0.86666667 0.93333333]
Average CV accuracy: 0.863225806451613


LogisticRegression()

🧪 **Kodunu test et**

In [187]:
from nbresult import ChallengeResult

result = ChallengeResult('base_model',
                         score = base_model_score
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/aybukealtuntas/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/aybukealtuntas/S16D2-S-Data-car-prices/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_base_model.py::TestBase_model::test_base_model_score PASSED         [100%]

============================== 1 passed in 0.12s ===============================


💯 You can commit your code:

git add tests/base_model.pickle

git commit -m 'Completed base_model step'

git push origin master



## (6) Özellik Seçimi  (Feature Selection (with _Permutation Importance_))

👩🏻‍🏫 Bir feature’ın target’ı tahmin etmede gerçekten önemli olup olmadığını tespit etmenin güçlü bir yolu şudur:

1. Bir model çalıştırın ve skorunu ölçün  
2. Bu feature’ı karıştırın (shuffle edin), modeli tekrar çalıştırın ve skoru tekrar ölçün  
    - Eğer performans **belirgin şekilde düşerse**, bu feature önemlidir ve **çıkarılmamalıdır**
    - Eğer performans **çok fazla düşmezse**, bu feature **elenebilir**

❓ **Sorular** ❓

1. Modele en az bilgi katkısı sağlayan feature’ları tespit etmek için feature permutation uygulayın.
2. Model performansının belirgin şekilde düşmeye başladığını fark edene kadar zayıf feature’ları dataset’ten çıkarın.
3. Elde ettiğiniz yeni güçlü feature set’i ile yeni bir modeli cross-validation ile değerlendirin ve skorunu `strong_model_score` adlı değişkende saklayın.

In [189]:
from sklearn.inspection import permutation_importance
import pandas as pd


# X_test ve y_test üzerinden önem skorlarını hesaplayalım
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)

# Skorları bir DataFrame'e dökerek görelim
importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': result.importances_mean
}).sort_values(by='importance_mean', ascending=False)

print(importance_df)

# Belirgin bir düşüş olana kadar (örneğin önemi 0.01'den küçük olanlar) özellikleri seçelim
# Eşik değeri (threshold) ihtiyaca göre ayarlayabiliriz
threshold = 0.005 
strong_features = importance_df[importance_df['importance_mean'] > threshold]['feature'].tolist()

print(f"Seçilen Güçlü Feature Sayısı: {len(strong_features)}")
print(f"Elenenler: {set(X_test.columns) - set(strong_features)}")

# Dataset'i sadece güçlü feature'lar kalacak şekilde güncelleyelim
X_train_strong = X_train[strong_features]
X_test_strong = X_test[strong_features]

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

# Yeni modeli oluşturalım (Parametreler base_model ile aynı olmalı ki farkı anlayalım)
strong_model = LogisticRegression()

# Cross-validation uygulayalım
cv_scores = cross_val_score(strong_model, X_train_strong, y_train, cv=5)

# İstenen değişken adıyla ortalama skoru saklayalım
strong_model_score = cv_scores.mean()

print(f"Güçlü Model Cross-Val Skoru: {strong_model_score}")

             feature  importance_mean
3         curbweight     2.794872e-01
2           carwidth     1.179487e-01
5             stroke     2.820513e-02
6            peakrpm     2.051282e-02
10    enginetype_ohc     1.794872e-02
0         aspiration     7.692308e-03
12   enginetype_ohcv     2.564103e-03
4     cylindernumber     2.564103e-03
1     enginelocation     0.000000e+00
8   enginetype_dohcv     0.000000e+00
9       enginetype_l     0.000000e+00
13  enginetype_rotor     0.000000e+00
7    enginetype_dohc    -2.220446e-17
11   enginetype_ohcf    -2.051282e-02
Seçilen Güçlü Feature Sayısı: 6
Elenenler: {'cylindernumber', 'enginetype_l', 'enginetype_dohcv', 'enginetype_ohcv', 'enginelocation', 'enginetype_dohc', 'enginetype_ohcf', 'enginetype_rotor'}
Güçlü Model Cross-Val Skoru: 0.8761290322580646


🧪 **Kodunu test et**

In [190]:
from nbresult import ChallengeResult

result = ChallengeResult('strong_model',
                         score = strong_model_score
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/aybukealtuntas/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/aybukealtuntas/S16D2-S-Data-car-prices/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_strong_model.py::TestStrong_model::test_strong_model_score PASSED   [100%]

============================== 1 passed in 0.18s ===============================


💯 You can commit your code:

git add tests/strong_model.pickle

git commit -m 'Completed strong_model step'

git push origin master



## Bonus -  Verilerinizi sınıflandırma (Stratifying your data) ⚖️

💡 Veriyi training ve testing olarak bölerken, dataset’imizdeki kategorik değişkenlerin oranına dikkat etmemiz gerekir — ister target `y`’nin sınıfları olsun ister `X` içindeki kategorik bir feature olsun.

Aşağıda bir örneğe bakalım 👇

❓ Orijinal `X` ve `y` verinizi sklearn’in `train_test_split` fonksiyonunu kullanarak training ve testing olarak ayırın; karşılaştırılabilir sonuçlar elde etmek için `random_state=1` ve `test_size=0.3` kullanın.

In [191]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=1)

❓ Training dataset’inizde ve testing dataset’inizde `price` sınıfı **1** olan araçların oranını kontrol edin.

> _Ham `df` içinde bu orana baktığınızda, yaklaşık **%50 / %50** olması gerekir._

In [198]:
print(y_train.value_counts())
print(y_test.value_counts())

price
1    67
0    66
Name: count, dtype: int64
price
1    30
0    28
Name: count, dtype: int64


☝️ Hâlâ yaklaşık olarak **%50 / %50** civarında olmalı.

***Peki random state’i değiştirirsek ne olur?***

❓ `random_state` değerlerini **1’den 10’a** kadar döngüye alın ve her seferinde training ve testing dataset’lerindeki `price` sınıfı **1** olan araçların oranını hesaplayın. ❓

In [204]:
random_state = range(1,10)
for a in random_state:
    print(f"{a} değerli random_state için")
    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=a)
    print((y_train == 1).mean())
    print((y_test == 1).mean())

1 değerli random_state için
0.5037593984962406
0.5172413793103449
2 değerli random_state için
0.48120300751879697
0.5689655172413793
3 değerli random_state için
0.5037593984962406
0.5172413793103449
4 değerli random_state için
0.5338345864661654
0.4482758620689655
5 değerli random_state için
0.5338345864661654
0.4482758620689655
6 değerli random_state için
0.49624060150375937
0.5344827586206896
7 değerli random_state için
0.5338345864661654
0.4482758620689655
8 değerli random_state için
0.48872180451127817
0.5517241379310345
9 değerli random_state için
0.5789473684210527
0.3448275862068966


Her seferinde oranların değiştiğini, hatta bazen oldukça ciddi şekilde değiştiğini gözlemleyeceksiniz 😱! Bu durum model performansını etkileyebilir.

❓ `train_test_split(random_state=1)` kullanılarak eğitilen bir Logistic Regression modelinin test skorunu,  
`random_state=9` kullanılarak eğitilen modelin test skoru ile karşılaştırın ❓

Eğitimi training data üzerinde yapmayı ve skoru testing data üzerinde hesaplamayı unutmayın.

In [205]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


# --- 1. Durum: random_state=1 ---
X_train1, X_test1, y_train1, y_test1 = train_test_split(X, y, test_size=0.2, random_state=1)
model.fit(X_train1, y_train1)
score1 = model.score(X_test1, y_test1)

# --- 2. Durum: random_state=9 ---
X_train9, X_test9, y_train9, y_test9 = train_test_split(X, y, test_size=0.2, random_state=9)
model.fit(X_train9, y_train9)
score9 = model.score(X_test9, y_test9)

# Karşılaştırma
print(f"Random State 1 Skoru: {score1:.4f}")
print(f"Random State 9 Skoru: {score9:.4f}")
print(f"Fark: {abs(score1 - score9):.4f}")

Random State 1 Skoru: 0.9487
Random State 9 Skoru: 0.9231
Fark: 0.0256


👀 `random_state=9` ile çok daha düşük bir skor görmelisiniz; çünkü bu test setindeki sınıf **1** araçların oranı %34.5 iken, training setinde bu oran %57.9’a, hatta orijinal dataset’te yaklaşık %50’ye yakındır.

Bu durum oldukça önemlidir; çünkü dataset’te oluşan bu **rastlantısal dengesizlik**, yalnızca model performansını düşürmekle kalmaz, aynı zamanda eğitim veya değerlendirme sırasında “gerçekliği” de bozabilir 🧐

***Peki bu sorunu nasıl çözebiliriz? Tren seti ve test seti arasında sınıfların dağılımını nasıl aynı tutabiliriz? 🔧***

🎁 Neyse ki sklearn’de, estimator (yani model) bir classifier olduğunda ve target bir sınıf olduğunda, bu durum `cross_validate` tarafından otomatik olarak ele alınır. 📚 [**sklearn.model_selection.cross_validate**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html) dokümantasyonunda `cv` parametresini inceleyin.

Çözüm, aşağıdakini kullanmaktır:

> 📚 [**Stratification (Katmanlama)**](https://scikit-learn.org/stable/modules/cross_validation.html#stratification)

### Hedefin tabakalaşması (Stratification of the target)

💡 ***Stratification*** tekniğini `train_test_split` içinde de kullanabiliriz.

❓ Bu kez **1’den 10’a** kadar olan `random_state` döngüsünü tekrar çalıştırın, ancak bu sefer holdout yöntemine ***`stratify=y`*** parametresini de ekleyin. ❓

In [206]:
random_state = range(1,10)
for a in random_state:
    print(f"{a} değerli random_state için")
    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=a,stratify=y)
    print((y_train == 1).mean())
    print((y_test == 1).mean())

1 değerli random_state için
0.5112781954887218
0.5
2 değerli random_state için
0.5112781954887218
0.5
3 değerli random_state için
0.5112781954887218
0.5
4 değerli random_state için
0.5112781954887218
0.5
5 değerli random_state için
0.5112781954887218
0.5
6 değerli random_state için
0.5112781954887218
0.5
7 değerli random_state için
0.5112781954887218
0.5
8 değerli random_state için
0.5112781954887218
0.5
9 değerli random_state için
0.5112781954887218
0.5


👀 Random state değişse bile, training ve testing verilerindeki sınıf oranları, orijinal `y` içindeki oranlarla aynı tutulur. İşte _stratification_ (katmanlama) tam olarak budur.

`train_test_split` fonksiyonunu `stratify` parametresiyle kullandığımızda, training ve testing verileri arasında **bir feature’ın oranlarını da koruyabiliriz**. Bu, özellikle aşağıdaki durumlarda son derece önemlidir:

- Churn tahmininde erkek ve kadın müşteri oranlarını korumak 🙋‍♂️ 🙋
- Ev fiyatlarını tahmin ederken büyük ve küçük evlerin oranlarını korumak 🏠 🏰
- Bir sonraki ürünü önerirken 1–5 arası review score dağılımını (multiclass!) korumak 🛍️
- vb.

Örneğin, bizim dataset’imizde `aspiration` feature’ının training ve testing verilerinde aynı oranda kalmasını istiyorsak, şu şekilde yazabiliriz:

`train_test_split(X, y, test_size=0.3, stratify=X.aspiration)`

---

Gördüğümüz gibi, **`cross_validate` [target değişkeni otomatik olarak stratify edebilir](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html#:~:text=For%20int/None%20inputs%2C%20if%20the%20estimator%20is%20a%20classifier%20and%20y%20is%20either%20binary%20or%20multiclass%2C%20StratifiedKFold%20is%20used.)**, ancak **feature’lar için bunu yapmaz** 🤔 Bunun için biraz ekstra çalışmaya ihtiyacımız var.

Bunun için `StratifiedKFold` kullanmamız gerekiyor 🔬

### Tabakalaşma (Stratification - generalized)

📚 [**StratifiedKFold**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html), veriyi `K` parçaya bölerken belirli sütunlar (feature veya target) üzerinden stratification yapmamıza olanak tanır.

Bu sayede, ilgilendiğimiz kategorik feature’ların oranlarını koruyarak manuel bir cross-validation yapabiliriz — bunu ikili (binary) `aspiration` feature’ı ile deneyelim:

In [207]:
from sklearn.model_selection import StratifiedKFold

# Veriyi 5 fold’a bölecek bir stratified k-fold oluşturma
skf = StratifiedKFold(n_splits=5)
scores = []

# .split() metodu bir iterator oluşturur; 'X.aspiration' stratify edeceğimiz feature’dır
for train_indices, test_indices in skf.split(X, X.aspiration):

    # 'train_indices' ve 'test_indices', orantılı bölünmeler üreten indeks listeleridir
    X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
    y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]

    # modeli başlatma ve eğitme
    model = LogisticRegression()
    model.fit(X_train, y_train)

    # en sonunda 5 fold’un ortalamasını almak için skoru listeye ekleme
    scores.append(model.score(X_test, y_test))

np.array(scores).mean()

0.8638326585695006

📚 [**StratifiedKFold**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html), veriyi `K` parçaya bölerken belirli sütunlar (feature veya target) üzerinden stratification yapmamıza olanak tanır.

Bu sayede, ilgilendiğimiz kategorik feature’ların oranlarını koruyarak manuel bir cross-validation yapabiliriz — bunu ikili (binary) `aspiration` feature’ı ile deneyelim:


🏁 Tebrikler! Tüm veri setini hazırladınız, özellik seçimi yaptınız ve hatta tabakalaşma hakkında bilgi edindiniz 💪.

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!